# 6) EE-3.1 Three-phase systems and per-unit values — Exercises

**Goals:** three-phase phasors, phase and line-to-line voltages, line currents of real equipment, three-phase losses, per-unit bases and base changes, and a two-voltage-level network with a transformer solved in per-unit and checked against pandapower.

Voltages quoted for three-phase equipment are line-to-line unless stated. S_base is three-phase. The helpers `power_flow_pu` (from the lecture) and `pandapower_two_level` are in the setup cell.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import math
import warnings

import numpy as np
import pandas as pd
import pandapower as pp

warnings.filterwarnings("ignore", category=FutureWarning)
A = np.exp(2j * np.pi / 3)          # 120-degree rotation operator


def numerical_jacobian(F, x, h=1e-7):
    """Central differences (M1.3)."""
    return np.column_stack([(F(x + e) - F(x - e)) / (2 * h) for e in np.eye(len(x)) * h])


def power_flow_pu(Y, S_inj, tol=1e-12):
    """Newton power flow in per-unit (from the lecture): bus 0 is the slack at 1.0 pu, all others PQ."""
    n = len(Y)
    def F(x):
        V = np.r_[1.0, x[n - 1:]] * np.exp(1j * np.r_[0.0, x[:n - 1]])
        d = V * np.conj(Y @ V) - S_inj
        return np.r_[d.real[1:], d.imag[1:]]
    x = np.r_[np.zeros(n - 1), np.ones(n - 1)]
    for _ in range(30):
        if np.abs(F(x)).max() < tol:
            return np.r_[1.0, x[n - 1:]] * np.exp(1j * np.r_[0.0, x[:n - 1]])
        x = x + np.linalg.solve(numerical_jacobian(F, x), -F(x))
    raise RuntimeError("no convergence")


def pandapower_two_level(trafo_mva=250.0, load_mva=120 + 40j):
    """The lecture's 400/110 kV network in pandapower, for checking."""
    net = pp.create_empty_network()
    b = [pp.create_bus(net, vn_kv=v) for v in [400, 400, 110, 110]]
    pp.create_ext_grid(net, b[0], vm_pu=1.0)
    pp.create_line_from_parameters(net, b[0], b[1], length_km=1, r_ohm_per_km=2, x_ohm_per_km=25, c_nf_per_km=0, max_i_ka=3)
    pp.create_transformer_from_parameters(net, b[1], b[2], sn_mva=trafo_mva, vn_hv_kv=400, vn_lv_kv=110, vk_percent=12,
                                          vkr_percent=0.3, pfe_kw=0, i0_percent=0, shift_degree=0)
    pp.create_line_from_parameters(net, b[2], b[3], length_km=1, r_ohm_per_km=1, x_ohm_per_km=4, c_nf_per_km=0, max_i_ka=3)
    pp.create_load(net, b[3], p_mw=load_mva.real, q_mvar=load_mva.imag)
    pp.runpp(net, numba=False)
    return net

### Warm-ups

**1. A balanced three-phase set**

Write `three_phase(V_ph, angle_deg=0)` returning the phasors `[Va, Vb, Vc]` (Vb lags Va by 120°).

In [2]:
def three_phase(V_ph: float, angle_deg: float = 0.0) -> np.ndarray:
    """Balanced phase voltages [Va, Vb, Vc] (phasors)."""
    ### BEGIN SOLUTION
    Va = V_ph * np.exp(1j * np.radians(angle_deg))
    return np.array([Va, Va * A**-1, Va * A])
    ### END SOLUTION

In [3]:
V = three_phase(230)
assert np.isclose(V.sum(), 0, atol=1e-9), "a balanced set sums to zero"
assert np.isclose(np.angle(V[1], deg=True), -120) and np.isclose(np.angle(V[2], deg=True), 120)
assert np.allclose(np.abs(three_phase(230, 30)), 230)

**2. Phase and line-to-line voltages**

Write `line_to_line(V_ph)` and `phase_voltage(V_LL)` (magnitudes). Then answer with code: what is the phase-to-ground voltage of a 400 kV line?

In [4]:
def line_to_line(V_ph: float) -> float:
    """Line-to-line voltage magnitude from the phase voltage."""
    ### BEGIN SOLUTION
    return math.sqrt(3) * V_ph
    ### END SOLUTION


def phase_voltage(V_LL: float) -> float:
    """Phase (line-to-neutral) voltage magnitude from the line-to-line voltage."""
    ### BEGIN SOLUTION
    return V_LL / math.sqrt(3)
    ### END SOLUTION

In [5]:
V = three_phase(230)
assert math.isclose(line_to_line(230), abs(V[0] - V[1])), "check against the phasor difference"
assert math.isclose(phase_voltage(line_to_line(230)), 230)
print(f"230 V phase -> {line_to_line(230):.1f} V line-to-line; 400 kV line -> {phase_voltage(400):.1f} kV to ground")

230 V phase -> 398.4 V line-to-line; 400 kV line -> 230.9 kV to ground


### Core

**3. Line currents of real equipment**

Write `line_current_A(P_kW, V_LL, pf=1.0)` for a balanced three-phase load, and `single_phase_current_A(P_kW, V_ph, pf=1.0)`. Use them for a 22 kW electric-vehicle charger.

In [6]:
def line_current_A(P_kW: float, V_LL: float, pf: float = 1.0) -> float:
    """Current (A) in each conductor of a balanced three-phase load."""
    ### BEGIN SOLUTION
    return P_kW * 1e3 / (math.sqrt(3) * V_LL * pf)
    ### END SOLUTION


def single_phase_current_A(P_kW: float, V_ph: float, pf: float = 1.0) -> float:
    """Current (A) of a single-phase load."""
    ### BEGIN SOLUTION
    return P_kW * 1e3 / (V_ph * pf)
    ### END SOLUTION

In [7]:
I3 = line_current_A(22, 400)
I1 = single_phase_current_A(22, 230)
assert math.isclose(I3, 31.75, rel_tol=1e-3), "a 22 kW wallbox draws about 32 A per phase"
assert math.isclose(I1 / I3, 3 * 400 / (math.sqrt(3) * 230), rel_tol=1e-9)
print(f"22 kW: three-phase {I3:.1f} A per conductor | single-phase {I1:.1f} A")

22 kW: three-phase 31.8 A per conductor | single-phase 95.7 A


**4. Three-phase line losses**

Write `line_losses_MW(P_MW, V_LL_kV, R_phase_ohm, pf=1.0)` = 3·I²·R with I the line current (kA). Show that it equals P²R / (V²·pf²) — the formula EE-1.1 used — and recompute EE-1.1's example: 100 MW over 10 Ω at 110 kV and at 400 kV.

In [8]:
def line_losses_MW(P_MW: float, V_LL_kV: float, R_phase_ohm: float, pf: float = 1.0) -> float:
    """Active losses (MW) of a three-phase line carrying P_MW."""
    ### BEGIN SOLUTION
    I_kA = P_MW / (math.sqrt(3) * V_LL_kV * pf)
    return 3 * I_kA**2 * R_phase_ohm
    ### END SOLUTION

In [9]:
for V_kV, expected in [(110, 8.26), (400, 0.62)]:
    loss = line_losses_MW(100, V_kV, 10)
    assert math.isclose(loss, 100**2 * 10 / V_kV**2), "3 I^2 R = P^2 R / V^2"
    assert round(loss, 2) == expected, "same numbers as EE-1.1"
    print(f"100 MW over 10 ohm at {V_kV} kV: {loss:.2f} MW lost")
assert line_losses_MW(100, 110, 10, pf=0.9) > line_losses_MW(100, 110, 10), "reactive power adds losses"

100 MW over 10 ohm at 110 kV: 8.26 MW lost
100 MW over 10 ohm at 400 kV: 0.62 MW lost


**5. Per-unit bases and base changes**

Write `bases(S_MVA, V_kV)` returning a dict with `Z_ohm` and `I_kA`, and `change_base(z_pu, S_old, S_new, V_old=1.0, V_new=1.0)` for z_new = z_old · (S_new/S_old) · (V_old/V_new)².

In [10]:
def bases(S_MVA: float, V_kV: float) -> dict:
    """Impedance base (ohm) and current base (kA) for a three-phase S base and line-to-line V base."""
    ### BEGIN SOLUTION
    return {"Z_ohm": V_kV**2 / S_MVA, "I_kA": S_MVA / (math.sqrt(3) * V_kV)}
    ### END SOLUTION


def change_base(z_pu: complex, S_old: float, S_new: float, V_old: float = 1.0, V_new: float = 1.0) -> complex:
    """Per-unit impedance on a new power and voltage base."""
    ### BEGIN SOLUTION
    return z_pu * (S_new / S_old) * (V_old / V_new) ** 2
    ### END SOLUTION

In [11]:
b = bases(100, 220)
assert math.isclose(b["Z_ohm"], 484) and math.isclose(b["I_kA"], 0.26243, rel_tol=1e-4)
assert math.isclose(change_base(0.12, 250, 100), 0.048), "the lecture's transformer on the system base"
assert math.isclose(change_base(0.2, 500, 100), 0.04), "a 500 MVA generator's 0.2 pu on 100 MVA"
z_ohm = 0.1 * bases(100, 110)["Z_ohm"]
assert math.isclose(change_base(0.1, 100, 100, 110, 115), z_ohm / bases(100, 115)["Z_ohm"]), "the same ohms on a new voltage base"

**6. Interpretation**

A large generator and a small one both have a reactance of 0.2 pu on their own rating. Which has the smaller impedance in ohms (at the same voltage), and why does that make sense physically? Why do engineers prefer to compare equipment in per-unit of its own rating?

<!-- BEGIN SOLUTION -->
Z_base = V²/S_rated, so the same 0.2 pu means 0.2·V²/S_rated ohms: the **larger** machine has the **smaller** impedance in ohms — task 5 shows the 500 MVA generator's 0.2 pu is only 0.04 pu on a 100 MVA base. Physically, a bigger machine is built to carry more current at the same voltage, which needs lower impedance (thicker conductors, larger cores); the relative voltage drop at rated current — which is what 0.2 pu describes — stays similar. That is the point of per-unit on the own rating: designs of very different sizes land in narrow, comparable ranges (transformers ≈ 0.08–0.15 pu, generator reactances ≈ 0.15–0.3 pu), so a value far outside the range immediately flags a data error, and one number tells you how "strong" the equipment is relative to its size.
<!-- END SOLUTION -->

### Challenge

**7. Two voltage levels with a transformer**

Write `two_level_network(trafo_mva, load_mva=120+40j, S_base=100)` that builds the lecture's network in per-unit (400 kV line 2 + j25 Ω, transformer 400/110 kV with v_k = 12 %, v_kr = 0.3 % on `trafo_mva`, 110 kV line 1 + j4 Ω), solves it with `power_flow_pu`, and returns a DataFrame with each bus's voltage in kV and in pu, plus a dict of active losses (MW) in the three branches. Check against pandapower for 250 MVA, then compare a 500 MVA transformer.

In [12]:
def two_level_network(trafo_mva: float, load_mva: complex = 120 + 40j, S_base: float = 100.0):
    """Bus voltages (DataFrame) and branch losses (dict, MW) of the 400/110 kV network."""
    ### BEGIN SOLUTION
    vk, vkr = 0.12, 0.003
    z = {"line 400 kV": (2 + 25j) / bases(S_base, 400)["Z_ohm"],
         "transformer": change_base(vkr + 1j * math.sqrt(vk**2 - vkr**2), trafo_mva, S_base),
         "line 110 kV": (1 + 4j) / bases(S_base, 110)["Z_ohm"]}
    ends = {"line 400 kV": (0, 1), "transformer": (1, 2), "line 110 kV": (2, 3)}
    Y = np.zeros((4, 4), dtype=complex)
    for name, (i, j) in ends.items():
        y = 1 / z[name]
        Y[i, i] += y; Y[j, j] += y; Y[i, j] -= y; Y[j, i] -= y
    V = power_flow_pu(Y, np.array([0, 0, 0, -load_mva / S_base]))
    v_nom = np.array([400, 400, 110, 110])
    buses = pd.DataFrame({"vm_pu": np.abs(V), "V_kV": np.abs(V) * v_nom, "angle_deg": np.angle(V, deg=True)})
    losses = {name: (abs((V[i] - V[j]) / z[name]) ** 2 * z[name]).real * S_base for name, (i, j) in ends.items()}
    return buses, losses
    ### END SOLUTION

In [13]:
buses, losses = two_level_network(250)
ref = pandapower_two_level(250)
assert np.allclose(buses.vm_pu, ref.res_bus.vm_pu, atol=1e-8) and np.allclose(buses.angle_deg, ref.res_bus.va_degree, atol=1e-6)
assert math.isclose(sum(losses.values()), ref.res_ext_grid.p_mw[0] - 120, rel_tol=1e-6), "losses = slack supply - load"
assert math.isclose(losses["transformer"], ref.res_trafo.pl_mw[0], rel_tol=1e-6), "transformer losses match pandapower"

buses_500, losses_500 = two_level_network(500)
ref_500 = pandapower_two_level(500)
assert np.allclose(buses_500.vm_pu, ref_500.res_bus.vm_pu, atol=1e-8)
assert buses_500.V_kV.iloc[3] > buses.V_kV.iloc[3], "a larger transformer (lower impedance) raises the load voltage"
print(f"250 MVA: load at {buses.V_kV.iloc[3]:.2f} kV, losses {sum(losses.values()):.3f} MW "
      f"| 500 MVA: load at {buses_500.V_kV.iloc[3]:.2f} kV, losses {sum(losses_500.values()):.3f} MW")
pd.DataFrame({"250 MVA": losses, "500 MVA": losses_500}).round(4)

250 MVA: load at 103.17 kV, losses 1.949 MW | 500 MVA: load at 104.82 kV, losses 1.782 MW


             250 MVA  500 MVA
line 400 kV   0.2273   0.2202
transformer   0.2182   0.1057
line 110 kV   1.5031   1.4561

**8. Interpretation**

The 500 MVA transformer has the same v_k = 12 % as the 250 MVA one. Why does it nevertheless raise the load voltage and change the losses? A planner wants the load voltage above 0.95 pu: what options does per-unit thinking suggest, and which element of the network would you change first?

<!-- BEGIN SOLUTION -->
The same 12 % refers to each transformer's **own** rating; on the 100 MVA system base the 500 MVA unit has half the impedance (0.024 vs. 0.048 pu), because a transformer built for twice the power carries twice the current with the same relative voltage drop. Less series impedance means less voltage drop and less I²R loss across it, so the load voltage rises and the transformer's losses fall. In per-unit the whole path is a chain of comparable series impedances, so the planner sees at once where the drop comes from: the transformer contributes the largest reactance, so reducing it (a larger or second parallel unit) or compensating it (adjusting the transformer's tap ratio to raise the secondary voltage, a capacitor bank at the 110 kV bus to supply the load's reactive power locally, EE-2.2) is usually more effective than reinforcing the 400 kV line, whose per-unit impedance is small. Tap changers are the cheapest first lever; the capacitor bank also reduces losses upstream.
<!-- END SOLUTION -->